In [ ]:
# Cell 1: Imports and Setup
import numpy as np
import pandas as pd
import time
import os
import sqlite3
from statistics import mean, stdev
from joblib import Parallel, delayed
from IPython.display import display

# Import your project files
from algo import metric_list
from foldrm import Classifier
from utils import split_data, split_xy, get_scores, count_rules_in_model, num_predicates, get_inverse_brier_score
from datasets import wine, ecoli, weight_lifting, wall_robot, page_blocks, nursery, dry_bean

# --- Experiment Configuration ---
datasets = [wine, ecoli, weight_lifting, wall_robot, page_blocks, nursery, dry_bean]
dataset_names = ["Wine", "Ecoli", "Weight Lifting", "Wall Robot", "Page Blocks", "Nursery", "Dry Bean"]

# Define the fit methods and strategies to test
fit_methods = ["FOLD-RM (fit)", "CON-FOLD (confidence_fit)"]
selection_strategies = ['greedy', 'round_robin', 'best_rule', 'info_gain']

# We will fix the gain metric to the default to isolate the effect of the selection strategy
#FIXED_METRIC = 'information_gain' 
metric_list = ['original', 'information_gain', 'Gini_Impurity', 'Precision_Information_Gain', 'Precision_Gini_Impurity']

NUM_TRIALS = 30 # Reduced for quicker testing; you can set this back to 300
DB_FILE = 'class_selection_experiment_2_30trials.db'

# Display options for the final DataFrame
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

# Cell 2: Helper Function for a Single Trial
def run_one_trial(dataset_name, strategy, fit_method, metric, data, model_template, num_classes, db_file):
    """
    Runs a single trial for a given selection strategy and fit method, and saves results to SQLite.
    """
    start_time = time.time()
    
    model = Classifier(attrs=model_template.attrs, numeric=model_template.numeric, label=model_template.label)
    data_train, data_test = split_data(data, ratio=0.8)
    X_test, Y_test = split_xy(data_test)
    
    # Conditionally call the correct fit method
    if fit_method == "CON-FOLD (confidence_fit)":
        model.confidence_fit(data_train, metric=metric, num_classes=num_classes, selection_strategy=strategy)
    else: # FOLD-RM (fit)
        model.fit(data_train, metric=metric, num_classes=num_classes, selection_strategy=strategy)
    
    Ystar_test_tuples = model.predict(X_test)
    Ystar_test = [y[0] for y in Ystar_test_tuples]
    score = get_scores(Ystar_test, data_test)
    brier_score = get_inverse_brier_score(Ystar_test_tuples, Y_test)
    
    rule_count = count_rules_in_model(model)
    
    model.asp()
    predicate_count = num_predicates(model)
    
    elapsed_time = time.time() - start_time
    
    # Save to database, now including fit_method
    conn = sqlite3.connect(db_file)
    cursor = conn.cursor()
    cursor.execute('PRAGMA journal_mode=WAL;')
    cursor.execute('''
        INSERT INTO trials (dataset, strategy, fit_method, metric, accuracy, brier_score, time, num_rules, num_preds)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', (dataset_name, strategy, fit_method, metric, score, brier_score, elapsed_time, rule_count, predicate_count))
    conn.commit()
    conn.close()

def run_experiments_for_combination(dataset_info, strategy, fit_method, metric, remaining_trials, db_file):
    """
    Runs the remaining trials for a specific dataset, strategy, and fit method in parallel.
    """
    dataset_func, name = dataset_info
    print(f"--- Starting: Dataset='{name}', Strategy='{strategy}', Fit Method='{fit_method}', 'Metric={metric}', Remaining Trials={remaining_trials} ---")
    
    model_template, data = dataset_func()
    num_classes = len(pd.unique(pd.DataFrame(data).iloc[:, -1]))
    
    if remaining_trials > 0:
        Parallel(n_jobs=-1, verbose=5)(
            delayed(run_one_trial)(name, strategy, fit_method, metric, data, model_template, num_classes, db_file)
            for _ in range(remaining_trials)
        )
    
    print(f"--- Finished: Dataset='{name}', Strategy='{strategy}', Fit Method='{fit_method}' ---")

# Cell 3: Main Experiment Runner with Smart Resume
if __name__ == "__main__":
    datasets_with_names = list(zip(datasets, dataset_names))

    # Create database and table if not exists
    conn = sqlite3.connect(DB_FILE)
    cursor = conn.cursor()
    # Add the fit_method column to the schema
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS trials (
            dataset TEXT,
            strategy TEXT,
            fit_method TEXT,
            metric TEXT,
            accuracy REAL,
            brier_score REAL,
            time REAL,
            num_rules INTEGER,
            num_preds INTEGER
        )
    ''')
    conn.commit()

    # Get current counts for each combination
    counts = {}
    cursor.execute('SELECT dataset, strategy, fit_method, metric, COUNT(*) FROM trials GROUP BY dataset, strategy, fit_method, metric')
    for row in cursor.fetchall():
        ds, strat, fm, met, cnt = row
        counts[(ds, strat, fm, met)] = cnt
    conn.close()

    # Create list of tasks to run, now iterating over fit_methods
    tasks_to_run = []
    for dataset_info in datasets_with_names:
        name = dataset_info[1]
        for strategy in selection_strategies:
            for fm in fit_methods:
                for metric in metric_list:
                    key = (name, strategy, fm, metric)
                    current_trials = counts.get(key, 0)
                    remaining = NUM_TRIALS - current_trials
                    if remaining > 0:
                        tasks_to_run.append({
                            'dataset_info': dataset_info,
                            'strategy': strategy,
                            'fit_method': fm,
                            'metric': metric,
                            'remaining_trials': remaining,
                            'db_file': DB_FILE
                        })

    print(f"\n>>> Found {len(tasks_to_run)} combinations with remaining trials. <<<\n")

    # Run tasks
    if tasks_to_run:
        for task in tasks_to_run:
            run_experiments_for_combination(**task)
        print("\n--- All tasks complete! You can now run the aggregation cell. ---")
    else:
        print("\n--- No remaining trials to run. Everything is complete. ---")

# Cell 4: Aggregate, Display, and Save Final Results
def process_and_display_results(df, fit_method_name, filename):
    """
    Helper function to pivot, average, format, display, and save results for a single fit method.
    """
    if df.empty:
        print(f"No completed data to process for {fit_method_name}.")
        return

    # 1. Group by dataset and strategy to get aggregates
    agg_df = df.groupby(['dataset', 'strategy']).agg({
        'accuracy': ['mean', 'std'], 'brier_score': ['mean', 'std'],
        'time': ['mean', 'std'], 'num_rules': ['mean', 'std'], 'num_preds': ['mean', 'std']
    })
    agg_df.columns = ['_'.join(col).strip() for col in agg_df.columns.values]
    agg_df.reset_index(inplace=True)

    # 2. Pivot the table for a nice comparison view
    pivoted = agg_df.pivot(index='dataset', columns='strategy')
    
    # 3. Reorder columns to group by metric (Accuracy, Brier, Time, etc.)
    metric_order = ['accuracy_mean', 'accuracy_std', 'brier_score_mean', 'brier_score_std', 
                    'time_mean', 'num_rules_mean', 'num_preds_mean']
    strategy_order = selection_strategies
    
    final_cols = [(metric, strategy) for metric in metric_order for strategy in strategy_order if (metric, strategy) in pivoted.columns]
    pivoted = pivoted.reindex(columns=final_cols)

    # 4. Add an 'Average' row across all datasets
    avg_row = agg_df.groupby('strategy').mean(numeric_only=True)
    avg_row.name = 'Average (All Datasets)'
    avg_row_pivoted = avg_row.unstack().to_frame().T
    avg_row_pivoted.index = [avg_row.name]
    avg_row_pivoted.columns = pd.MultiIndex.from_tuples(avg_row_pivoted.columns)
    
    final_summary = pd.concat([pivoted, avg_row_pivoted])

    # 5. Display and Save
    print(f"\n{'='*40}")
    print(f"--- STRATEGY EXPERIMENT SUMMARY: {fit_method_name} ---")
    
    styled_df = final_summary.style.format("{:.4f}", na_rep="-") \
        .background_gradient(cmap='viridis', axis=1, 
                             subset=[(m, s) for m in ['accuracy_mean', 'brier_score_mean'] for s in strategy_order])

    display(styled_df)
    
    final_summary.to_csv(filename)
    print(f"\nSummary results for {fit_method_name} saved to {filename}")
    print(f"{'='*40}")


if __name__ == "__main__":
    backup_file = 'strategy_experiment_2_30trials_results_full.csv'

    # 1. Load all trial data from the database
    conn = sqlite3.connect(DB_FILE)
    try:
        all_trials_df = pd.read_sql_query("SELECT * FROM trials", conn)
    except pd.io.sql.DatabaseError:
        all_trials_df = pd.DataFrame()
    conn.close()

    if not all_trials_df.empty:
        # 2. Get trial counts and filter for completed experiments
        trial_counts = all_trials_df.groupby(['dataset', 'strategy', 'fit_method', 'metric']).size().reset_index(name='trial_count')
        completed_combinations = trial_counts[trial_counts['trial_count'] >= NUM_TRIALS]
        
        # Merge back to get only the data from completed runs
        completed_df = pd.merge(all_trials_df, completed_combinations, on=['dataset', 'strategy', 'fit_method', 'metric'])

        if not completed_df.empty:
            # Save a backup of all completed trial data
            completed_df.to_csv(backup_file, index=False)
            print(f"Backup of all completed trial data saved to {backup_file}")
        
            # Get the list of metrics that have completed trials
            metrics_to_process = completed_df['metric'].unique()
        
            # Loop through each metric and generate a separate report
            for metric in metrics_to_process:
                print(f"\n\n{'#'*80}")
                print(f"### PROCESSING RESULTS FOR METRIC: {metric} ###")
                print(f"{'#'*80}")
                
                metric_df = completed_df[completed_df['metric'] == metric]
        
                # Process and display results for FOLD-RM for this metric
                fold_rm_df = metric_df[metric_df['fit_method'] == 'FOLD-RM (fit)'].copy()
                fold_rm_filename = f"fold_rm_strategy_summary_{metric}.csv"
                process_and_display_results(fold_rm_df, f"FOLD-RM (fit) - Metric: {metric}", fold_rm_filename)
        
                # Process and display results for CON-FOLD for this metric
                con_fold_df = metric_df[metric_df['fit_method'] == 'CON-FOLD (confidence_fit)'].copy()
                con_fold_filename = f"con_fold_strategy_summary_{metric}.csv"
                process_and_display_results(con_fold_df, f"CON-FOLD (confidence_fit) - Metric: {metric}", con_fold_filename)
        else:
            print("--- No combinations are fully complete yet (>= " + str(NUM_TRIALS) + " trials). ---")
    else:
        print("--- No trial data found in the database. ---")

C:\Users\Lachlan McGinness\GithubRepositories\CONFOLD\algo.py:2: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.3.3)
  from scipy.stats import binom



>>> Found 280 combinations with remaining trials. <<<

--- Starting: Dataset='Wine', Strategy='greedy', Fit Method='FOLD-RM (fit)', 'Metric=original', Remaining Trials=30 ---

% wine dataset (178, 14)


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of  30 | elapsed:    2.7s remaining:   39.7s
[Parallel(n_jobs=-1)]: Done   9 out of  30 | elapsed:    3.0s remaining:    7.1s
[Parallel(n_jobs=-1)]: Done  16 out of  30 | elapsed:    3.1s remaining:    2.7s
[Parallel(n_jobs=-1)]: Done  23 out of  30 | elapsed:    3.2s remaining:    0.9s
[Parallel(n_jobs=-1)]: Done  30 out of  30 | elapsed:    3.3s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 32 concurrent workers.


--- Finished: Dataset='Wine', Strategy='greedy', Fit Method='FOLD-RM (fit)' ---
--- Starting: Dataset='Wine', Strategy='greedy', Fit Method='FOLD-RM (fit)', 'Metric=information_gain', Remaining Trials=30 ---

% wine dataset (178, 14)


[Parallel(n_jobs=-1)]: Done   2 out of  30 | elapsed:    0.6s remaining:    9.4s
[Parallel(n_jobs=-1)]: Done   9 out of  30 | elapsed:    0.6s remaining:    1.6s
[Parallel(n_jobs=-1)]: Done  16 out of  30 | elapsed:    0.7s remaining:    0.6s
[Parallel(n_jobs=-1)]: Done  23 out of  30 | elapsed:    0.7s remaining:    0.1s
[Parallel(n_jobs=-1)]: Done  30 out of  30 | elapsed:    0.8s finished


...

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of  30 | elapsed: 47.6min remaining: 665.9min
[Parallel(n_jobs=-1)]: Done   9 out of  30 | elapsed: 53.8min remaining: 125.5min
[Parallel(n_jobs=-1)]: Done  16 out of  30 | elapsed: 65.1min remaining: 56.9min
[Parallel(n_jobs=-1)]: Done  23 out of  30 | elapsed: 68.9min remaining: 21.0min
[Parallel(n_jobs=-1)]: Done  30 out of  30 | elapsed: 75.8min finished


--- Finished: Dataset='Dry Bean', Strategy='best_rule', Fit Method='FOLD-RM (fit)' ---
--- Starting: Dataset='Dry Bean', Strategy='best_rule', Fit Method='FOLD-RM (fit)', 'Metric=information_gain', Remaining Trials=30 ---

% dry bean dataset (13611, 17)


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of  30 | elapsed:   29.6s remaining:  6.9min
[Parallel(n_jobs=-1)]: Done   9 out of  30 | elapsed:   30.6s remaining:  1.2min
[Parallel(n_jobs=-1)]: Done  16 out of  30 | elapsed:   31.2s remaining:   27.3s
[Parallel(n_jobs=-1)]: Done  23 out of  30 | elapsed:   49.8s remaining:   15.1s
[Parallel(n_jobs=-1)]: Done  30 out of  30 | elapsed:   54.4s finished


--- Finished: Dataset='Dry Bean', Strategy='best_rule', Fit Method='FOLD-RM (fit)' ---
--- Starting: Dataset='Dry Bean', Strategy='best_rule', Fit Method='FOLD-RM (fit)', 'Metric=Gini_Impurity', Remaining Trials=30 ---

% dry bean dataset (13611, 17)


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 out of  30 | elapsed: 343.1min remaining: 4803.1min
[Parallel(n_jobs=-1)]: Done   9 out of  30 | elapsed: 1345.2min remaining: 3138.7min
[Parallel(n_jobs=-1)]: Done  16 out of  30 | elapsed: 1877.3min remaining: 1642.7min
[Parallel(n_jobs=-1)]: Done  23 out of  30 | elapsed: 2066.3min remaining: 628.9min
